## 0. Installation et imports

In [ ]:
# Sur Kaggle : décommenter la ligne suivante si ultralytics n'est pas déjà installé
!pip installq ultralytics

In [ ]:
import os, glob, time, json, math, subprocess, warnings
from collections import deque, Counter
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

try:
    from IPython.display import Video, display
except Exception:                      # exécution hors Jupyter
    Video = display = None

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./results"
os.makedirs(OUT_DIR, exist_ok=True)
print("OpenCV :", cv2.__version__, "| sortie :", OUT_DIR)

## 1. Paramètres du système
Toutes les hypothèses sont regroupées ici : elles sont **modifiables** et **discutées dans le rapport**.
Les valeurs imposées par l'énoncé (f, hauteurs, seuils, décélération) sont repérées par `[ÉNONCÉ]`, les autres sont des hypothèses d'ingénieur `[HYP]`.

In [ ]:
# ───────────── Caméra (modèle sténopé) ─────────────
F_PX = 720.0                      # [ÉNONCÉ] focale équivalente (px)

# ───────────── Classes COCO utiles et hauteurs réelles typiques (m) ─────────────
# 0 person, 1 bicycle, 2 car, 3 motorcycle, 5 bus, 7 truck
CLASSES = {
    0: ("pieton",   1.70),        # [ÉNONCÉ]
    1: ("cycliste", 1.70),        # [HYP] cycliste + vélo
    2: ("voiture",  1.50),        # [ÉNONCÉ]
    3: ("moto",     1.50),        # [HYP] moto + pilote
    5: ("bus",      3.00),        # [HYP] 3,0–3,2 m
    7: ("camion",   3.00),        # [HYP] très dispersé (pick-up 1,9 m … semi-remorque 4 m)
}

# ───────────── Détection / suivi ─────────────
MODEL_NAME = "yolov8s.pt"         # n = plus rapide, s = bon compromis, m = plus précis
CONF  = 0.30
IMGSZ = 960                       # une résolution plus grande aide les objets lointains
MIN_H_PX = 6                      # boîtes plus petites ignorées

# ───────────── Incertitude de mesure de h_px (propagée vers Z) ─────────────
SIG_H0, SIG_H_REL = 1.0, 0.02     # [HYP] σ_h = 1 px + 2 % de h_px  (jitter de la boîte YOLO)
R_MIN = 0.3**2                    # plancher de variance sur Z (m²)

# ───────────── Filtre de Kalman (un par cible) ─────────────
SIGMA_A  = 2.0                    # [HYP] écart-type de l'accélération relative (m/s²)
SIGMA_V0 = 15.0                   # incertitude initiale sur la vitesse relative (m/s)
GATE_NIS = 9.0                    # rejet si innovation² / S > 9  (≈ 3σ)
GATE_WARMUP = 5                   # pas de rejet pendant les 5 premières mesures (la vitesse n'est pas encore connue)
REJECT_MAX = 3                    # après 3 rejets consécutifs : réinitialisation
Z_MIN, Z_MAX = 0.3, 100.0         # plage de validité du modèle sténopé
MIN_AGE, V_STD_MAX, V_STD_REL = 4, 3.0, 0.25   # cible « confirmée » : ≥4 mesures et σ_v ≤ max(3 m/s ; 25 % de v_rel)
MAX_MISS, MAX_MISS_VALID = 8, 3   # images sans détection : suppression / validité
WIN_S = 0.6                       # fenêtre (s) de la régression pondérée de 1/h
V_MIN = 0.5                       # vitesse de rapprochement minimale pour définir un TTC (m/s)
TTC_MODE = "kalman"               # "kalman" | "regression" | "min"

# ───────────── Couloir de collision (géométrie métrique) ─────────────
EGO_HALF_W   = 0.95               # demi-largeur du véhicule ego (m)
LAT_MARGIN   = 0.25               # marge latérale (m)
HALF_W_MAX   = 1.30               # plafonne la demi-largeur estimée de la cible (m)
LANE_OFFSET_PX = 0.0              # décalage du centre optique par rapport à l'axe véhicule (px)

# ───────────── Machine à états ─────────────
TTC_FCW, TTC_AEB = 1.6, 0.8       # [ÉNONCÉ] seuils (s)
A_AEB = -8.5                      # [ÉNONCÉ] décélération commandée (m/s²)
HYST = 0.1                        # hystérésis de désactivation (s)
N_CONFIRM = 2                     # anti-rebond : images consécutives avant de monter de niveau
HOLD_AEB_S = 1.0                  # maintien minimal du freinage (s) ; en pratique max(1 s, temps d'arrêt + 0,5 s)
ADAPTIVE_AEB = False              # True : seuil AEB dépendant de la vitesse (extension, cf. §6)
ADAPT_MARGIN = 0.25               # marge de sécurité (s) ajoutée au TTC minimal d'évitement (mode adaptatif)

# ───────────── Actionneur (utilisé en simulation) ─────────────
T_LAT, T_PROC, T_RAMP = 0.15, 0.05, 0.20   # [HYP] hydraulique, calcul, montée en pression (s)
T_DELAY = T_LAT + T_PROC

## 2. Étape 1 – Perception (YOLOv8 + ByteTrack)
Le détecteur renvoie une liste d'objets `Det` (classe, boîte `[x1,y1,x2,y2]`, confiance, identifiant de piste).
La hauteur apparente est `h_px = y2 − y1`. L'interface `detector(frame) -> list[Det]` permet de changer de modèle sans toucher au reste.

In [ ]:

from dataclasses import dataclass
from typing import Optional


@dataclass
class Det:
    cls: int
    x1: float
    y1: float
    x2: float
    y2: float
    conf: float = 1.0
    tid: Optional[int] = None

    @property
    def h(self):
        """Hauteur apparente de la cible en pixels."""
        return self.y2 - self.y1


class YoloDetector:
    """YOLOv8 avec suivi ByteTrack optionnel."""

    def __init__(
        self,
        weights=None,
        conf=None,
        imgsz=None,
        track=True
    ):
        import torch
        from ultralytics import YOLO

        # Chargement du modèle
        self.model = YOLO(weights or MODEL_NAME)

        # Paramètres de détection
        self.conf = CONF if conf is None else conf
        self.imgsz = IMGSZ if imgsz is None else imgsz
        self.track = track

        # Sélection automatique du périphérique
        self.device = 0 if torch.cuda.is_available() else "cpu"

        device_name = "GPU" if self.device != "cpu" else "CPU"

        self.name = (
            f"{weights or MODEL_NAME} "
            f"({device_name}, imgsz={self.imgsz})"
        )

    def __call__(self, frame):
        """Détecte les objets et retourne une liste de Det."""

        # Paramètres communs : aucune option half
        kw = {
            "classes": list(CLASSES),
            "conf": self.conf,
            "imgsz": self.imgsz,
            "device": self.device,
            "verbose": False,
        }

        # Détection avec ou sans suivi
        if self.track:
            res = self.model.track(
                frame,
                persist=True,
                tracker="bytetrack.yaml",
                **kw
            )[0]
        else:
            res = self.model.predict(
                frame,
                **kw
            )[0]

        # Récupération des boîtes de détection
        b = res.boxes

        if b is None or len(b) == 0:
            return []

        xyxy = b.xyxy.cpu().numpy()
        cls = b.cls.cpu().numpy().astype(int)
        cf = b.conf.cpu().numpy()

        # Identifiants de suivi ByteTrack
        if b.id is not None:
            ids = b.id.cpu().numpy().astype(int)
        else:
            ids = [None] * len(cls)

        # Conversion vers les objets Det
        return [
            Det(
                cls=int(c),
                x1=float(bb[0]),
                y1=float(bb[1]),
                x2=float(bb[2]),
                y2=float(bb[3]),
                conf=float(p),
                tid=int(i) if i is not None else None,
            )
            for c, bb, p, i in zip(cls, xyxy, cf, ids)
        ]


## 3. Étape 2 – Modèle physique : distance monoculaire et estimation d'état

### 3.1 Sténopé et propagation d'incertitude
$$Z=\frac{f\,H}{h_{px}}\qquad\Rightarrow\qquad \sigma_Z=\left|\frac{\partial Z}{\partial h}\right|\sigma_h=\frac{Z^2}{f\,H}\,\sigma_h\qquad\Rightarrow\qquad \frac{\sigma_Z}{Z}=\frac{\sigma_h}{h_{px}}$$
L'erreur sur Z croît **quadratiquement** avec la distance : le bruit de mesure du filtre ne peut donc pas être constant. La variance de mesure $R=\sigma_Z^2$ est recalculée **à chaque image**.

### 3.2 Deux estimateurs du TTC
* **Kalman** (vitesse constante) : état $x=[Z,\dot Z]^T$, $TTC=Z/v_{rel}$ avec $v_{rel}=-\dot Z$.
* **Régression pondérée de 1/h** : à vitesse de rapprochement constante, $1/h=(Z_0-v\,t)/(fH)$ est **linéaire** en $t$. Le TTC est $y(t_{now})/(-\dot y)$ : il **ne dépend ni de f ni de H**, et chaque point est pondéré par $1/\sigma_y^2$ avec $\sigma_y=\sigma_h/h^2$.

> **Remarque importante.** Z dépend de H (une erreur de 20 % sur H donne 20 % d'erreur sur Z), mais **le TTC = Z/v_rel est invariant** : H se simplifie dans le rapport. L'erreur de hauteur affecte l'affichage de la distance, pas (au premier ordre) le déclenchement.

In [ ]:
def pinhole_distance(h_px, H_real, f=F_PX):
    # Z = f * H_reelle / h_px
    return f * H_real / max(h_px, 1e-6)


def sigma_h(h_px):
    return SIG_H0 + SIG_H_REL * h_px


class Kalman1D:
    # Filtre de Kalman discret, modèle à vitesse constante (bruit d'accélération blanc), mesure = Z.
    def __init__(self, dt, sigma_a=None):
        sa = SIGMA_A if sigma_a is None else sigma_a
        self.dt = dt
        self.F = np.array([[1.0, dt], [0.0, 1.0]])
        self.Hm = np.array([[1.0, 0.0]])
        self.Q = sa**2 * np.array([[dt**4/4, dt**3/2], [dt**3/2, dt**2]])
        self.x = None; self.P = None

    def init(self, z, r):
        self.x = np.array([[z], [0.0]])
        self.P = np.diag([r, SIGMA_V0**2])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def nis(self, z, r):
        # innovation normalisée au carré (suit un χ² à 1 ddl si le filtre est cohérent)
        S = float(self.P[0, 0]) + r
        return (z - float(self.x[0, 0]))**2 / S

    def update(self, z, r):
        S = float(self.P[0, 0]) + r
        y = z - float(self.x[0, 0])
        K = self.P[:, 0:1] / S
        self.x = self.x + K * y
        IKH = np.eye(2) - K @ self.Hm
        self.P = IKH @ self.P @ IKH.T + (K * r) @ K.T       # forme de Joseph (stable)


def ttc_regression(hist, min_pts=5):
    # TTC par régression pondérée de y = 1/h sur le temps (indépendant de f et H).
    if len(hist) < min_pts:
        return np.inf
    a = np.asarray(hist, float)
    t = a[:, 0] - a[-1, 0]; h = a[:, 1]
    y = 1.0 / h
    w = 1.0 / (sigma_h(h) / h**2) ** 2
    W = w.sum(); tb = (w * t).sum() / W; yb = (w * y).sum() / W
    Stt = (w * (t - tb)**2).sum()
    if Stt <= 0:
        return np.inf
    b = (w * (t - tb) * (y - yb)).sum() / Stt
    a0 = yb - b * tb                      # valeur ajustée de 1/h à l'instant courant
    se_b = 1.0 / np.sqrt(Stt)
    if b >= -2 * se_b or a0 <= 0:         # rapprochement non significatif à 2σ
        return np.inf
    return a0 / (-b)


class TargetEstimator:
    # Estimateur dédié à UNE piste : Kalman adaptatif + régression de 1/h.
    def __init__(self, tid, cls_id, dt, fps):
        self.tid, self.dt = tid, dt
        self.votes = deque([cls_id], maxlen=15)
        self.cls, self.H = cls_id, CLASSES[cls_id][1]
        self.kf = Kalman1D(dt)
        self.hist = deque(maxlen=max(5, int(round(WIN_S * fps))))
        self.age = 0; self.miss = 0; self.rej = 0
        self.last_nis = np.nan; self.last_det = None; self.used = False

    def _vote(self, cls_id):
        # classe majoritaire sur 15 images ; si elle change, Z et Ż sont remis à l'échelle (continuité)
        self.votes.append(cls_id)
        top = max(set(self.votes), key=self.votes.count)
        if top != self.cls and len(self.votes) >= 5 and self.votes.count(top) >= 0.6 * len(self.votes):
            ratio = CLASSES[top][1] / self.H
            if self.kf.x is not None:
                S = np.diag([ratio, ratio]); self.kf.x = S @ self.kf.x; self.kf.P = S @ self.kf.P @ S.T
            self.cls, self.H = top, CLASSES[top][1]

    def step(self, t, h_px=None, cls_id=None, truncated=False):
        if cls_id is not None:
            self._vote(cls_id)
        if self.kf.x is not None:
            self.kf.predict()
        self.used = False; self.last_nis = np.nan
        if h_px is not None and h_px >= MIN_H_PX and not truncated:
            Z = pinhole_distance(h_px, self.H)
            r = max((Z * Z * sigma_h(h_px) / (F_PX * self.H))**2, R_MIN)
            if self.kf.x is None:
                self.kf.init(Z, r); self.age = 1; self.hist.append((t, h_px)); self.used = True
            else:
                nis = self.kf.nis(Z, r); self.last_nis = nis
                if nis <= GATE_NIS or self.age < GATE_WARMUP:
                    self.kf.update(Z, r); self.age += 1; self.rej = 0
                    self.hist.append((t, h_px)); self.used = True
                else:
                    self.rej += 1
                    if self.rej >= REJECT_MAX:             # rupture de modèle : on repart de la mesure
                        self.kf.init(Z, r); self.hist.clear(); self.hist.append((t, h_px))
                        self.age = 1; self.rej = 0; self.used = True
        self.miss = 0 if self.used else self.miss + 1
        if self.miss > MAX_MISS:
            self.hist.clear()

    def state(self):
        if self.kf.x is None:
            return None
        Z = float(self.kf.x[0, 0]); vz = float(self.kf.x[1, 0])
        vstd = float(np.sqrt(max(self.kf.P[1, 1], 0.0)))
        vrel = -vz
        ttc_kf = Z / vrel if vrel > V_MIN else np.inf
        ttc_rg = ttc_regression(self.hist)
        confirmed = (self.age >= MIN_AGE and vstd <= max(V_STD_MAX, V_STD_REL * abs(vrel)) and Z_MIN <= Z <= Z_MAX
                     and self.miss <= MAX_MISS_VALID)
        return dict(Z=Z, vrel=vrel, vstd=vstd, ttc_kf=ttc_kf, ttc_reg=ttc_rg, confirmed=confirmed)

## 4. Validation Monte-Carlo de l'estimation (vérité terrain connue)
Il n'existe pas de LiDAR dans la vidéo : on **ne peut pas mesurer l'erreur de Z sur la vidéo**. On valide donc l'estimateur sur des approches **synthétiques** dont la distance vraie est connue, avec le **bruit de boîte simulé 30 % plus fort** que celui supposé par le filtre (test de robustesse à un modèle de bruit imparfait) et avec un **biais de hauteur de 20 %** (cible plus haute que prévu).

In [ ]:
def synth_h(Z, H_true, rng, mismatch=1.0):
    h = F_PX * H_true / Z
    return h + rng.normal() * mismatch * sigma_h(h)


def run_approach(Z0, v_rel, H_true, rng, fps=30, mismatch=1.3, Z_end=3.0):
    # approche à vitesse relative constante ; renvoie vérité et estimations image par image
    dt = 1.0 / fps
    n = int((Z0 - Z_end) / v_rel * fps)
    est = TargetEstimator(1, 2, dt, fps)               # voiture : H supposée = 1,50 m
    rows, Zprev = [], None
    for k in range(n):
        Zt = Z0 - v_rel * k * dt
        h = synth_h(Zt, H_true, rng, mismatch)
        est.step(k * dt, h, 2, False)
        s = est.state()
        Zraw = pinhole_distance(h, CLASSES[2][1])
        vr = (Zprev - Zraw) / dt if Zprev is not None else 0.0   # dérivée brute sur 1 image
        ttc_naive = Zraw / vr if vr > V_MIN else np.inf
        Zprev = Zraw
        rows.append((k * dt, Zt, Zt / v_rel, Zraw, s["Z"], s["ttc_kf"], s["ttc_reg"], ttc_naive, s["confirmed"]))
    return pd.DataFrame(rows, columns=["t", "Z_true", "ttc_true", "Z_raw", "Z_kf", "ttc_kf", "ttc_reg", "ttc_naive", "ok"])


def monte_carlo(n_trials=60, v_list=(5, 10, 15), h_bias=(1.0, 1.2), seed=42):
    rng = np.random.default_rng(seed); out = []
    for hb in h_bias:
        for v in v_list:
            for _ in range(n_trials):
                d = run_approach(Z0=70.0, v_rel=v, H_true=CLASSES[2][1] * hb, rng=rng)
                d["v_rel"], d["h_bias"] = v, hb
                out.append(d)
    return pd.concat(out, ignore_index=True)

t0 = time.time()
MC = monte_carlo()
print(f"Monte-Carlo : {len(MC):,} échantillons en {time.time()-t0:.0f} s")

In [ ]:
# ── Tableau : erreur sur le TTC à quelques TTC vrais clés (|ε| en secondes) ──
def ttc_table(MC, h_bias=1.0, centers=(3.0, 2.0, 1.6, 0.8), half=0.1):
    rows = []
    sub = MC[(MC.h_bias == h_bias) & MC.ok]
    for c in centers:
        s = sub[(sub.ttc_true - c).abs() <= half]
        for name in ("ttc_kf", "ttc_reg", "ttc_naive"):
            e = (s[name] - s.ttc_true).abs()
            fin = np.isfinite(e)
            rows.append(dict(TTC_vrai=c, estimateur=name, valide_pct=100 * fin.mean(),
                             erreur_P50=e[fin].median(), erreur_P90=e[fin].quantile(.9)))
    return pd.DataFrame(rows).round(3)

print("=== Hauteur supposée exacte (H_vrai = 1,50 m) ===")
display_fn = display if display else print
T1 = ttc_table(MC, 1.0); display_fn(T1)
print("=== Cible 20 % plus haute que supposé (H_vrai = 1,80 m) : Z faux de ~20 %, mais TTC ? ===")
T2 = ttc_table(MC, 1.2); display_fn(T2)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
# (a) erreur relative de Z : théorie σ_h/h vs simulation
Zg = np.linspace(5, 100, 60); hg = F_PX * 1.5 / Zg
ax[0].plot(Zg, 100 * sigma_h(hg) / hg, "k-", label="théorie σ_h / h")
s = MC[(MC.h_bias == 1.0) & MC.ok]; s = s.assign(zb=(s.Z_true // 5) * 5 + 2.5)
g = s.groupby("zb").apply(lambda d: 100 * ((d.Z_raw - d.Z_true) / d.Z_true).std())
ax[0].plot(g.index, g.values, "o", label="Monte-Carlo (bruit ×1,3)")
ax[0].set_xlabel("Distance vraie Z (m)"); ax[0].set_ylabel("σ_Z / Z (%)"); ax[0].set_title("Précision du sténopé"); ax[0].legend()
# (b) biais de H : Z faux, TTC juste
sb = MC[(MC.h_bias == 1.2) & MC.ok & (MC.Z_true > 10)]
ax[1].hist(100 * ((sb.Z_kf - sb.Z_true) / sb.Z_true), bins=60, color="tab:red", alpha=.8)
ax[1].set_title("H sous-estimée de 20 % → erreur sur Z"); ax[1].set_xlabel("erreur relative Z (%)")
e = (sb.ttc_kf - sb.ttc_true) / sb.ttc_true; e = e[np.isfinite(e)]
ax[2].hist(100 * e.clip(-100, 100), bins=60, color="tab:green", alpha=.8)
ax[2].set_title("… mais erreur relative sur le TTC (Kalman)"); ax[2].set_xlabel("erreur relative TTC (%)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_montecarlo.png", bbox_inches="tight"); plt.show()

In [ ]:
# ── Erreur du TTC en fonction du TTC vrai (v_rel = 10 m/s) : médiane et bande P10–P90 ──
s = MC[(MC.h_bias == 1.0) & (MC.v_rel == 10) & MC.ok].copy()
s["bin"] = (s.ttc_true / 0.25).round() * 0.25
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for name, col, lab in (("ttc_naive", "tab:gray", "dérivée brute (1 image)"),
                       ("ttc_kf", "tab:red", "Kalman adaptatif"),
                       ("ttc_reg", "tab:blue", "régression pondérée 1/h")):
    r = (s[name] - s.ttc_true).replace([np.inf, -np.inf], np.nan)
    g = r.groupby(s.bin)
    ax.plot(g.median().index, g.median().values, color=col, label=lab)
    ax.fill_between(g.quantile(.1).index, g.quantile(.1).values, g.quantile(.9).values, color=col, alpha=.15)
ax.axhline(0, color="k", lw=.6); ax.set_xlim(0.5, 4); ax.set_ylim(-1.5, 1.5)
ax.axvline(TTC_AEB, color="r", ls="--", lw=.8); ax.axvline(TTC_FCW, color="orange", ls="--", lw=.8)
ax.set_xlabel("TTC vrai (s)"); ax.set_ylabel("erreur TTC estimé − vrai (s)")
ax.set_title("Erreur sur le TTC (bruit de boîte ×1,3 vs modèle du filtre)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_ttc_error.png", bbox_inches="tight"); plt.show()

## 5. Étape 3 – TTC et machine à états finis
$TTC=Z/v_{rel}$ avec $v_{rel}=v_{ego}-v_{cible}$ (vitesse de **rapprochement**) ; si $v_{rel}\le v_{min}$ il n'y a pas de collision : $TTC=\infty$.

La machine à états applique les seuils de l'énoncé et ajoute trois protections contre le **freinage fantôme** :
1. **hystérésis** à la descente (on ne revient en arrière que si TTC > seuil + 0,1 s) ;
2. **anti-rebond** : il faut `N_CONFIRM` images consécutives au niveau supérieur (compromis : 2 images = 67 ms de latence à 30 fps) ;
3. **maintien** : un freinage enclenché est maintenu au moins le **temps d'arrêt** $v_{rel}/|a|+T_{delay}+0{,}5$ s. Sans cela, la cible sort du champ de la caméra à courte distance (boîte tronquée), le TTC devient infini et le freinage serait relâché *avant l'arrêt* (défaut mis en évidence par la simulation §7).

In [ ]:
NOMINAL, FCW, AEB = 0, 1, 2
STATE_NAME  = {NOMINAL: "NOMINAL", FCW: "FCW", AEB: "AEB"}
STATE_COLOR = {NOMINAL: (0, 200, 0), FCW: (0, 165, 255), AEB: (0, 0, 255)}     # BGR : vert / orange / rouge
STATE_CMD   = {NOMINAL: "ACC regulation normale",
               FCW:     "ALERTE + pre-remplissage hydraulique",
               AEB:     f"FREINAGE URGENCE a={A_AEB} m/s2"}


def compute_ttc(Z, v_rel, v_min=None):
    v_min = V_MIN if v_min is None else v_min
    return Z / v_rel if v_rel > v_min else np.inf


class AEBStateMachine:
    def __init__(self, dt, n_confirm=None, hold_s=None):
        self.dt = dt
        self.n_confirm = N_CONFIRM if n_confirm is None else n_confirm
        self.hold_n = int(round((HOLD_AEB_S if hold_s is None else hold_s) / dt))
        self.state, self.cnt, self.hold = NOMINAL, 0, 0

    @staticmethod
    def raw_level(ttc, t_aeb=None, t_fcw=None):
        t_aeb = TTC_AEB if t_aeb is None else t_aeb
        t_fcw = TTC_FCW if t_fcw is None else t_fcw
        if ttc <= t_aeb: return AEB
        if ttc <= t_fcw: return FCW
        return NOMINAL

    def update(self, ttc, t_aeb=None, t_fcw=None, vrel=None):
        t_aeb = TTC_AEB if t_aeb is None else t_aeb
        t_fcw = TTC_FCW if t_fcw is None else t_fcw
        raw = self.raw_level(ttc, t_aeb, t_fcw)
        if raw > self.state:                              # montée : anti-rebond
            self.cnt += 1
            if self.cnt >= self.n_confirm:
                self.state, self.cnt = raw, 0
                if raw == AEB:                            # maintien ≥ temps d'arrêt estimé
                    t_stop = (vrel / abs(A_AEB) + T_DELAY + 0.5) if (vrel is not None and vrel > 0) else 0.0
                    self.hold = max(self.hold_n, int(round(t_stop / self.dt)))
                else:
                    self.hold = 0
        else:
            self.cnt = 0
            if self.hold > 0:                             # maintien du freinage
                self.hold -= 1
            elif raw < self.state:                        # descente : hystérésis
                upper = {AEB: t_aeb, FCW: t_fcw}[self.state]
                if ttc > upper + HYST:
                    self.state = raw
        return self.state


# ── Tests unitaires de la FSM ──
f = AEBStateMachine(1/30, n_confirm=1, hold_s=0)                       # comportement « énoncé + hystérésis »
seq = [3.0, 1.5, 0.9, 0.7, 0.85, 0.95, 1.5, 1.8]
got = [STATE_NAME[f.update(t)] for t in seq]
assert got == ["NOMINAL", "FCW", "FCW", "AEB", "AEB", "FCW", "FCW", "NOMINAL"], got
f = AEBStateMachine(1/30, n_confirm=2, hold_s=0)                       # un pic isolé ne déclenche rien
assert [f.update(t) for t in [3.0, 0.5, 3.0, 3.0]] == [NOMINAL] * 4
f = AEBStateMachine(1/30, n_confirm=2, hold_s=1.0)                     # maintien 1 s
for t in [0.5, 0.5]: f.update(t)
assert f.state == AEB and all(f.update(5.0) == AEB for _ in range(25)) and f.state == AEB
assert [f.update(5.0) for _ in range(10)][-1] == NOMINAL
assert compute_ttc(10, 0.2) == np.inf and abs(compute_ttc(10, 5) - 2) < 1e-12
print("Tests FSM : OK  (séquence énoncé, anti-rebond, maintien, TTC infini)")

## 6. Système complet : suivi multi-cibles → couloir de collision → TTC critique → FSM → commande
Pour chaque image : (1) mise à jour de l'estimateur de **chaque piste**, (2) calcul de la position latérale réelle $X=(x_c-c_x)\,Z/f$, (3) test d'appartenance au **couloir de collision** (largeur du véhicule ego + demi-largeur de la cible + marge), (4) sélection de la **cible critique** = TTC minimal parmi les cibles *confirmées* dans le couloir, (5) FSM, (6) commande de décélération.

**Mode adaptatif (extension, désactivé par défaut) :** le seuil AEB devient `max(0,8 s ; d_arrêt(v_rel)/v_rel + marge)` où `d_arrêt` est la distance de freinage avec latence et rampe (§7). Il montre ce qu'il faut changer pour *éviter* et non seulement *atténuer*.

In [ ]:
def stopping_distance(v, a=None, t_delay=None, t_ramp=None):
    # distance parcourue de la détection à l'arrêt : retard pur + rampe de décélération linéaire + palier
    A = abs(A_AEB if a is None else a); td = T_DELAY if t_delay is None else t_delay
    tr = T_RAMP if t_ramp is None else t_ramp
    d = v * td
    if v <= A * tr / 2:                                   # arrêt avant la fin de la rampe
        ts = np.sqrt(2 * v * tr / A); return d + v * ts - A * ts**3 / (6 * tr)
    d += v * tr - A * tr**2 / 6
    return d + (v - A * tr / 2)**2 / (2 * A)


def ttc_min_avoid(v_rel):
    # TTC minimal au déclenchement pour éviter la collision contre un obstacle immobile (repère relatif)
    return stopping_distance(v_rel) / max(v_rel, 1e-6)


def in_ego_path(x1, x2, Z, W):
    # le centre de la boîte est converti en décalage latéral métrique ; collision si recouvrement avec le couloir ego
    cx = W / 2 + LANE_OFFSET_PX
    Xc = ((x1 + x2) / 2 - cx) * Z / F_PX
    half_w = min((x2 - x1) * Z / F_PX / 2, HALF_W_MAX)
    return abs(Xc) - half_w <= EGO_HALF_W + LAT_MARGIN


class AEBSystem:
    def __init__(self, W, H, fps, adaptive=None):
        self.W, self.H, self.fps, self.dt = W, H, fps, 1.0 / fps
        self.adaptive = ADAPTIVE_AEB if adaptive is None else adaptive
        self.tracks, self.fsm, self.k = {}, AEBStateMachine(self.dt), 0

    def _ttc(self, s):
        if TTC_MODE == "regression": return s["ttc_reg"]
        if TTC_MODE == "min": return min(s["ttc_kf"], s["ttc_reg"])
        return s["ttc_kf"]

    def thresholds(self, vrel):
        if not self.adaptive or vrel is None or vrel <= V_MIN:
            return TTC_AEB, TTC_FCW
        t_aeb = min(max(TTC_AEB, ttc_min_avoid(vrel) + ADAPT_MARGIN + (N_CONFIRM - 1) * self.dt), 2.5)   # marge : anti-rebond inclus
        return t_aeb, max(TTC_FCW, t_aeb + 0.6)

    def process(self, dets):
        t = self.k * self.dt; self.k += 1
        seen = set()
        for d in dets:                                     # 1) mise à jour des pistes vues
            if d.tid is None or d.cls not in CLASSES:
                continue
            est = self.tracks.get(d.tid)
            if est is None:
                est = self.tracks[d.tid] = TargetEstimator(d.tid, d.cls, self.dt, self.fps)
            truncated = d.y2 >= self.H - 2 or d.y1 <= 1   # boîte coupée par le bord → h_px sous-estimée → Z surestimée
            est.step(t, d.h, d.cls, truncated); est.last_det = d; est.trunc = truncated; seen.add(d.tid)
        for tid in list(self.tracks):                      # 2) pistes non vues : prédiction seule
            if tid not in seen:
                est = self.tracks[tid]; est.step(t)
                if est.miss > MAX_MISS:
                    del self.tracks[tid]

        targets = []
        for tid, est in self.tracks.items():               # 3) état de chaque cible
            s = est.state()
            if s is None or est.last_det is None:
                continue
            d = est.last_det
            s.update(tid=tid, cls=est.cls, name=CLASSES[est.cls][0], box=(d.x1, d.y1, d.x2, d.y2), h_px=d.h,
                     conf=d.conf, coasting=(tid not in seen), nis=est.last_nis,
                     in_path=in_ego_path(d.x1, d.x2, s["Z"], self.W) if s["Z"] > 0 else True)
            s["ttc"] = self._ttc(s)
            targets.append(s)

        cand = [s for s in targets if s["confirmed"] and s["in_path"]]       # 4) cible critique
        crit = min(cand, key=lambda s: s["ttc"]) if cand else None
        if crit is not None and not np.isfinite(crit["ttc"]):
            crit = min(cand, key=lambda s: s["Z"])          # aucune collision prévue : on affiche la plus proche
        ttc = crit["ttc"] if crit is not None else np.inf
        t_aeb, t_fcw = self.thresholds(crit["vrel"] if crit is not None else None)
        for s in targets:                                   # niveau de chaque cible (couleur)
            s["level"] = self.fsm.raw_level(s["ttc"], t_aeb, t_fcw) if (s["confirmed"] and s["in_path"]) else None
        state = self.fsm.update(ttc, t_aeb, t_fcw, crit["vrel"] if crit is not None else None)   # 5) FSM
        a_cmd = A_AEB if state == AEB else 0.0              # 6) Agir
        return dict(t=t, state=state, ttc=ttc, a_cmd=a_cmd, critical=crit, targets=targets, t_aeb=t_aeb, t_fcw=t_fcw)

## 7. Faisabilité physique et simulation dynamique en boucle fermée
Un système AEB ne se juge pas seulement à sa détection, mais à sa **capacité à arrêter le véhicule**. En supposant un obstacle immobile, la distance minimale pour éviter la collision est
$$Z_{min}(v)=v\,T_{delay}+\Big(v\,T_r-\tfrac{|a|T_r^2}{6}\Big)+\frac{\big(v-|a|T_r/2\big)^2}{2|a|}\qquad\Rightarrow\qquad TTC_{min}^{évit}=\frac{Z_{min}(v)}{v}$$
avec $T_{delay}=T_{lat}+T_{proc}$ (hydraulique + calcul) et $T_r$ la rampe de montée de la décélération. On compare ensuite la FSM **conforme à l'énoncé** (0,8 s) au **mode adaptatif**, avec le **même code** de perception que sur la vidéo (la boîte englobante est synthétisée par le modèle de caméra, avec bruit).

In [ ]:
v_kmh = np.arange(10, 131, 10); v = v_kmh / 3.6
tab = pd.DataFrame({"v_rel (km/h)": v_kmh,
                    "d_arrêt (m)": [stopping_distance(x) for x in v],
                    "TTC_min évitement (s)": [ttc_min_avoid(x) for x in v]}).round(2)
(display if display else print)(tab.set_index("v_rel (km/h)").T)
v_lim = next(x for x in np.arange(1, 60, .05) if ttc_min_avoid(x) > TTC_AEB)
print(f"Avec un seuil fixe de {TTC_AEB} s, l'évitement complet d'un obstacle immobile n'est possible que jusqu'à ≈ {v_lim*3.6:.0f} km/h (freinage idéal, perception parfaite).")
print("Au-delà, l'AEB à seuil fixe ATTÉNUE le choc (réduit la vitesse d'impact) mais ne l'évite pas.")

In [ ]:
SIM_W, SIM_H, SIM_YH, SIM_HCAM, SIM_WCAR = 1280, 720, 360.0, 1.3, 1.8

def make_det(Z, H_true, rng, noise=True, mismatch=1.0):
    # boîte englobante synthétique (caméra à 1,3 m, horizon à mi-hauteur), cible dans l'axe
    h = F_PX * H_true / Z
    if noise: h += rng.normal() * mismatch * sigma_h(h)
    y2 = SIM_YH + F_PX * SIM_HCAM / Z; w = F_PX * SIM_WCAR / Z
    return Det(2, SIM_W/2 - w/2, y2 - h, SIM_W/2 + w/2, y2, 0.9, 1)


def simulate(v_ego_kmh, Z0=80.0, v_tgt_kmh=0.0, tgt_brake=None, adaptive=False, noise=True,
             mismatch=1.0, seed=0, H_true=1.5, fps=30, T_max=25.0, log=False):
    # Boucle fermée : perception bruitée → AEBSystem → retard + rampe → dynamique longitudinale. Conducteur passif.
    rng = np.random.default_rng(seed); dt = 1.0 / fps
    sys_ = AEBSystem(SIM_W, SIM_H, fps, adaptive=adaptive)
    v_e, v_t, Z, a = v_ego_kmh / 3.6, v_tgt_kmh / 3.6, Z0, 0.0
    J = abs(A_AEB) / T_RAMP; nd = int(round(T_DELAY / dt)); cmd_log = []
    trig, rows, collided, v_imp = None, [], False, 0.0
    for k in range(int(T_max * fps)):
        t = k * dt
        a_t = tgt_brake[1] if (tgt_brake and t >= tgt_brake[0] and v_t > 0) else 0.0
        res = sys_.process([make_det(Z, H_true, rng, noise, mismatch)])
        cmd_log.append(1 if res["state"] == AEB else 0)
        if res["state"] == AEB and trig is None:
            trig = dict(t=t, Z=Z, ttc=Z / max(v_e - v_t, 1e-6), v_rel=v_e - v_t)
        cmd_d = cmd_log[k - nd] if k - nd >= 0 else 0
        a += np.clip((-abs(A_AEB) if cmd_d else 0.0) - a, -J * dt, J * dt)
        v_e2, v_t2 = max(v_e + a * dt, 0.0), max(v_t + a_t * dt, 0.0)
        Z -= ((v_e + v_e2) / 2 - (v_t + v_t2) / 2) * dt
        v_e, v_t = v_e2, v_t2
        if log:
            c = res["critical"]
            rows.append((t, Z, c["Z"] if c else np.nan, res["ttc"], v_e * 3.6, v_t * 3.6, a, res["state"]))
        if Z <= 0:
            collided, v_imp = True, (v_e - v_t) * 3.6; break
        if trig is not None and v_e <= v_t + 1e-3:
            break
    out = dict(collision=collided, v_impact_kmh=v_imp, gap_final_m=max(Z, 0.0), declenchement=trig)
    if log: out["log"] = pd.DataFrame(rows, columns=["t", "Z_true", "Z_est", "ttc_est", "v_ego_kmh", "v_tgt_kmh", "a", "state"])
    return out


rows = []
for v0 in (30, 50, 70, 90, 110):
    for mode, ad in (("énoncé (0,8 s)", False), ("adaptatif", True)):
        rs = [simulate(v0, adaptive=ad, seed=s, mismatch=1.3) for s in range(8)]
        trg = [r["declenchement"] for r in rs if r["declenchement"]]
        rows.append({"v_ego (km/h)": v0, "mode": mode,
                     "collisions (sur 8)": sum(r["collision"] for r in rs),
                     "v_impact moy. (km/h)": np.mean([r["v_impact_kmh"] for r in rs]),
                     "TTC vrai au déclenchement (s)": np.mean([x["ttc"] for x in trg]) if trg else np.nan,
                     "Z au déclenchement (m)": np.mean([x["Z"] for x in trg]) if trg else np.nan,
                     "TTC_min évitement (s)": ttc_min_avoid(v0 / 3.6)})
SIM_TAB = pd.DataFrame(rows).round(2)
(display if display else print)(SIM_TAB)
SIM_TAB.to_csv(f"{OUT_DIR}/simulation_boucle_fermee.csv", index=False)

In [ ]:
# ── Visualisation : obstacle immobile à 50 km/h, FSM énoncé vs adaptatif ; et freinage brusque du véhicule de tête ──
fig, ax = plt.subplots(2, 3, figsize=(16, 7), sharex="col")
for col, (title, kw) in enumerate((("Obstacle immobile, ego 70 km/h", dict(v_ego_kmh=70)),
                                   ("Obstacle immobile, ego 50 km/h", dict(v_ego_kmh=50)),
                                   ("Suiveur 50 km/h à 22 m, tête freine à −7 m/s²", dict(v_ego_kmh=50, v_tgt_kmh=50, Z0=22, tgt_brake=(2.0, -7.0))))):
    for ad, c in ((False, "tab:red"), (True, "tab:green")):
        r = simulate(adaptive=ad, mismatch=1.3, log=True, seed=3, **kw); L = r["log"]
        lab = ("adaptatif" if ad else "énoncé") + (f" → choc {r['v_impact_kmh']:.0f} km/h" if r["collision"] else " → évité")
        ax[0, col].plot(L.t, L.Z_true.clip(lower=0), color=c, label=lab)
        ax[1, col].plot(L.t, L.v_ego_kmh, color=c)
    ax[0, col].set_title(title, fontsize=10); ax[0, col].legend(fontsize=8); ax[0, col].set_ylabel("distance vraie Z (m)")
    ax[1, col].set_ylabel("v_ego (km/h)"); ax[1, col].set_xlabel("temps (s)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_simulation.png", bbox_inches="tight"); plt.show()

## 8. Étape 4 – Pipeline vidéo, rendu graphique et mesure du temps réel
Rendu : cadre de couleur dynamique (vert / orange / rouge), distance Z, TTC, **jauge de TTC**, **courbe glissante de TTC** et état de commande du bloc de freinage. Le temps est mesuré **par étage** ; la latence « ECU » = détection + estimation/décision (le rendu et l'écriture vidéo n'existeraient pas dans une voiture).

In [ ]:
def _panel(frame, x, y, w, h, alpha=0.55):
    sub = frame[y:y+h, x:x+w]
    cv2.addWeighted(np.zeros_like(sub), alpha, sub, 1 - alpha, 0, sub)


def draw_overlay(frame, res, ttc_hist, ecu_ms):
    H, W = frame.shape[:2]
    state = res["state"]; col = STATE_COLOR[state]; crit = res["critical"]
    for s in res["targets"]:
        if s["coasting"]:
            continue
        x1, y1, x2, y2 = map(int, s["box"])
        lvl = s["level"]
        if lvl is None:                                    # hors couloir ou non confirmée : discret
            cv2.rectangle(frame, (x1, y1), (x2, y2), (150, 150, 150), 1); continue
        c = STATE_COLOR[lvl]; thick = 4 if (crit is not None and s["tid"] == crit["tid"]) else 2
        cv2.rectangle(frame, (x1, y1), (x2, y2), c, thick)
        ttc_txt = "inf" if not np.isfinite(s["ttc"]) else f"{s['ttc']:.2f}s"
        label = f"#{s['tid']} {s['name']} Z={s['Z']:.1f}m TTC={ttc_txt}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(frame, (x1, max(y1 - th - 10, 0)), (x1 + tw + 6, max(y1, th + 10)), c, -1)
        cv2.putText(frame, label, (x1 + 3, max(y1 - 5, th + 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    # bandeau supérieur
    _panel(frame, 0, 0, W, 72, 0.75)
    cv2.putText(frame, f"ETAT : {STATE_NAME[state]}", (15, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.95, col, 2)
    cv2.putText(frame, STATE_CMD[state], (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    vr = crit["vrel"] if crit is not None else 0.0
    Zc = f"{crit['Z']:.1f} m" if crit is not None else "--"
    cv2.putText(frame, f"Z={Zc}  v_rel={vr:.1f} m/s  a_cmd={res['a_cmd']:.1f} m/s2", (W - 560, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(frame, f"latence ECU={ecu_ms:.1f} ms  ({1000/max(ecu_ms,1e-3):.0f} FPS)", (W - 560, 58),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    # jauge de TTC (0–4 s)
    gx, gy, gw, gh = 20, H - 40, 360, 16; sc = gw / 4.0
    cv2.rectangle(frame, (gx, gy), (gx + int(TTC_AEB*sc), gy + gh), STATE_COLOR[AEB], -1)
    cv2.rectangle(frame, (gx + int(TTC_AEB*sc), gy), (gx + int(TTC_FCW*sc), gy + gh), STATE_COLOR[FCW], -1)
    cv2.rectangle(frame, (gx + int(TTC_FCW*sc), gy), (gx + gw, gy + gh), STATE_COLOR[NOMINAL], -1)
    cv2.rectangle(frame, (gx, gy), (gx + gw, gy + gh), (255, 255, 255), 1)
    if np.isfinite(res["ttc"]):
        mx = gx + int(min(res["ttc"], 4.0) * sc)
        cv2.drawMarker(frame, (mx, gy - 4), (255, 255, 255), cv2.MARKER_TRIANGLE_DOWN, 14, 2)
    cv2.putText(frame, "TTC (s)  0 .. 4", (gx, gy - 26), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    # courbe glissante du TTC (4 dernières secondes)
    px, py, pw, ph = W - 340, H - 130, 320, 110
    _panel(frame, px, py, pw, ph, 0.6)
    for thr, c in ((TTC_FCW, STATE_COLOR[FCW]), (TTC_AEB, STATE_COLOR[AEB])):
        yy = py + ph - int(min(thr, 4.0) / 4.0 * ph); cv2.line(frame, (px, yy), (px + pw, yy), c, 1)
    pts = [(px + int(i * pw / max(len(ttc_hist) - 1, 1)), py + ph - int(min(v, 4.0) / 4.0 * ph))
           for i, v in enumerate(ttc_hist)]
    if len(pts) > 1:
        cv2.polylines(frame, [np.array(pts, np.int32)], False, (255, 255, 255), 2)
    cv2.putText(frame, "TTC critique (4 s)", (px + 5, py + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return frame


def run_pipeline(video_path, detector, out_path=None, max_frames=None, render=True, adaptive=None, max_snaps=6):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"Impossible d'ouvrir la vidéo : {video_path}"
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if (fps and fps == fps and fps >= 1) else 30.0
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_path = out_path or f"{OUT_DIR}/aeb_output.mp4"
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H)) if render else None
    system = AEBSystem(W, H, fps, adaptive)
    ttc_hist, ecu_win = deque(maxlen=int(4 * fps)), deque(maxlen=30)
    rows, trows, snaps, prev_state, k = [], [], [], NOMINAL, 0
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames and k >= max_frames):
            break
        t0 = time.perf_counter(); dets = detector(frame)
        t1 = time.perf_counter(); res = system.process(dets)
        t2 = time.perf_counter()
        ecu_win.append((t2 - t0) * 1000)
        ttc_hist.append(res["ttc"] if np.isfinite(res["ttc"]) else 4.0)
        t_draw = t_write = 0.0
        if render:
            td = time.perf_counter()
            frame = draw_overlay(frame, res, ttc_hist, float(np.mean(ecu_win)))
            t_draw = (time.perf_counter() - td) * 1000
            tw = time.perf_counter(); writer.write(frame); t_write = (time.perf_counter() - tw) * 1000
            if res["state"] > prev_state and len(snaps) < max_snaps:
                snaps.append((k, k / fps, res["state"], frame.copy()))
        prev_state = res["state"]
        c = res["critical"]
        rows.append(dict(frame=k, t=k / fps, state=res["state"], ttc=res["ttc"], a_cmd=res["a_cmd"],
                         Z=c["Z"] if c else np.nan, v_rel=c["vrel"] if c else np.nan,
                         ttc_kf=c["ttc_kf"] if c else np.nan, ttc_reg=c["ttc_reg"] if c else np.nan,
                         tid=c["tid"] if c else -1, cls=c["name"] if c else "", n_det=len(dets),
                         t_det_ms=(t1 - t0) * 1000, t_est_ms=(t2 - t1) * 1000, t_draw_ms=t_draw, t_write_ms=t_write,
                         t_ecu_ms=(t2 - t0) * 1000))
        for s in res["targets"]:
            trows.append(dict(frame=k, t=k / fps, tid=s["tid"], cls=s["name"], Z=s["Z"], v_rel=s["vrel"],
                              v_std=s["vstd"], ttc_kf=s["ttc_kf"], ttc_reg=s["ttc_reg"], confirmed=s["confirmed"],
                              in_path=s["in_path"], h_px=s["h_px"], nis=s["nis"], coasting=s["coasting"]))
        k += 1
    cap.release()
    if writer: writer.release()
    meta = dict(fps=fps, W=W, H=H, n_frames=k, n_total=n_tot, out_path=out_path)
    return pd.DataFrame(rows), pd.DataFrame(trows), snaps, meta

In [ ]:
%pip install -q ultralytics
%pip install -q "lap>=0.5.12"

### Lancement sur la vidéo `00067cfb-e535423e.mov`
Le code cherche **d'abord** ce fichier précis (la version de base ne cherchait que `.mp4` / `.avi` et aurait ignoré un `.mov`), puis n'importe quelle vidéo.

In [ ]:
#@video
def find_video(name="00067cfb-e535423e"):
    exts = ("mov", "mp4", "avi", "mkv", "MOV", "MP4")
    allv = sorted(sum([glob.glob(f"/kaggle/input/**/*.{e}", recursive=True) for e in exts], []))
    pref = [p for p in allv if name in os.path.basename(p)]
    return (pref or allv or [None])[0], allv

VIDEO_PATH, ALL_VIDEOS = find_video()
print("Vidéos trouvées :", ALL_VIDEOS[:10])
assert VIDEO_PATH, "Ajoutez la dataset « Video mini projet adas » via Add Data."
cap = cv2.VideoCapture(VIDEO_PATH)
print(f"Vidéo : {VIDEO_PATH}\n  {int(cap.get(3))}x{int(cap.get(4))} px, {cap.get(5):.2f} fps, {int(cap.get(7))} images")
cap.release()

detector = YoloDetector()
print("Détecteur :", detector.name)
df, df_t, snaps, meta = run_pipeline(VIDEO_PATH, detector, f"{OUT_DIR}/aeb_output.mp4")
print(f"{meta['n_frames']} images traitées à {meta['fps']:.0f} fps nominal")
df.head()

In [ ]:
#@video
# Ré-encodage H.264 pour lecture dans le navigateur
r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f"{OUT_DIR}/aeb_output.mp4",
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", f"{OUT_DIR}/aeb_output_h264.mp4"])
if r.returncode == 0 and display:
    display(Video(f"{OUT_DIR}/aeb_output_h264.mp4", embed=True, width=800))

## 9. Résultats
### 9.1 Courbes de la cible critique

In [ ]:
#@video
fig, ax = plt.subplots(5, 1, figsize=(11, 15), sharex=True)
ax[0].plot(df.t, df.Z, "r-"); ax[0].set_ylabel("Z cible critique (m)")
ax[1].plot(df.t, df.v_rel, "g-"); ax[1].axhline(0, color="k", lw=.5); ax[1].set_ylabel("v_rel (m/s)")
cl = lambda s: s.replace(np.inf, np.nan).clip(upper=6)
ax[2].plot(df.t, cl(df.ttc_kf), "r-", label="TTC Kalman"); ax[2].plot(df.t, cl(df.ttc_reg), "b-", alpha=.6, label="TTC régression 1/h")
ax[2].axhline(TTC_FCW, color="orange", ls="--", label="FCW 1,6 s"); ax[2].axhline(TTC_AEB, color="red", ls="--", label="AEB 0,8 s")
ax[2].set_ylabel("TTC (s)"); ax[2].legend(ncol=4, fontsize=8)
ax[3].step(df.t, df.state, where="post", color="purple"); ax[3].set_yticks([0, 1, 2]); ax[3].set_yticklabels(["NOMINAL", "FCW", "AEB"])
ax[3].set_ylabel("État FSM")
ax[4].plot(df.t, df.n_det, color="gray"); ax[4].set_ylabel("nb de détections"); ax[4].set_xlabel("Temps (s)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_courbes_video.png", bbox_inches="tight"); plt.show()
print("Répartition des états (% des images) :")
print((df.state.map(STATE_NAME).value_counts(normalize=True) * 100).round(1))

### 9.2 Événements FCW / AEB et vérification visuelle
Chaque passage en FCW ou AEB est listé avec le TTC minimal, la distance et la cible concernée. **Sur une vidéo de conduite normale, tout AEB est a priori un faux positif** : les instantanés ci-dessous permettent de le vérifier à l'œil et d'analyser la cause (cible hors voie, boîte tronquée, changement de classe…).

In [ ]:
#@video
def extract_events(df):
    ev, cur = [], None
    for r in df.itertuples():
        if r.state >= FCW:
            if cur is None: cur = dict(t_debut=r.t, niveau_max=r.state, rows=[])
            cur["niveau_max"] = max(cur["niveau_max"], r.state); cur["rows"].append(r)
        elif cur is not None:
            ev.append(cur); cur = None
    if cur: ev.append(cur)
    out = []
    for e in ev:
        d = pd.DataFrame(e["rows"]); i = d.ttc.idxmin()
        out.append(dict(t_debut=round(e["t_debut"], 2), duree_s=round(d.t.iloc[-1] - d.t.iloc[0] + 1/meta["fps"], 2),
                        niveau_max=STATE_NAME[e["niveau_max"]], TTC_min_s=round(d.ttc.min(), 2),
                        Z_au_min_m=round(d.Z[i], 1), v_rel_ms=round(d.v_rel[i], 1), cible=f"#{d.tid[i]} {d.cls[i]}"))
    return pd.DataFrame(out)

EV = extract_events(df)
(display if display else print)(EV if len(EV) else "Aucun événement FCW/AEB sur cette vidéo.")
EV.to_csv(f"{OUT_DIR}/evenements_fcw_aeb.csv", index=False)

if snaps:
    n = len(snaps); fig, ax = plt.subplots((n + 1) // 2, 2, figsize=(14, 4.2 * ((n + 1) // 2)), squeeze=False)
    for a in ax.ravel(): a.axis("off")
    for a, (k, t, st, fr) in zip(ax.ravel(), snaps):
        a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); a.set_title(f"t = {t:.2f} s (image {k}) → passage en {STATE_NAME[st]}", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_evenements.png", bbox_inches="tight"); plt.show()

### 9.3 Cohérence interne de l'estimation (sans vérité terrain)
Trois indicateurs **mesurés sur la vidéo** :
* **NIS moyen** des innovations du filtre : il doit valoir ≈ 1 (χ² à 1 ddl). ≫ 1 : bruit sous-estimé ; ≪ 1 : bruit surestimé.
* **Taux de rejet** du test de cohérence (mesures aberrantes écartées).
* **Accord entre les deux TTC** (Kalman vs régression de 1/h, indépendants de H et f).

In [ ]:
#@video
nis = df_t.nis.dropna()
conf = df_t[df_t.confirmed & df_t.in_path & ~df_t.coasting]
both = conf[np.isfinite(conf.ttc_kf) & np.isfinite(conf.ttc_reg) & (conf.ttc_kf < 6) & (conf.ttc_reg < 6)]
rel = ((both.ttc_kf - both.ttc_reg).abs() / both[["ttc_kf", "ttc_reg"]].mean(axis=1))
METRICS = {
    "images_traitees": int(meta["n_frames"]),
    "pistes_distinctes": int(df_t.tid.nunique()),
    "NIS_moyen": float(nis.mean()), "NIS_P95": float(nis.quantile(.95)),
    "taux_rejet_mesures_%": float(100 * (nis > GATE_NIS).mean()),
    "TTC_accord_kalman_regression_mediane_%": float(100 * rel.median()) if len(rel) else None,
    "TTC_accord_P90_%": float(100 * rel.quantile(.9)) if len(rel) else None,
    "echantillons_compares": int(len(both)),
}
for k_, v_ in METRICS.items(): print(f"{k_:42s} {v_}")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(nis.clip(upper=12), bins=50, color="steelblue", density=True); ax[0].axvline(1, color="r", ls="--"); ax[0].set_title("NIS (cible : moyenne ≈ 1)")
ax[1].scatter(both.ttc_reg, both.ttc_kf, s=3, alpha=.4); ax[1].plot([0, 6], [0, 6], "r--"); ax[1].set_xlabel("TTC régression 1/h (s)"); ax[1].set_ylabel("TTC Kalman (s)"); ax[1].set_title("Accord des deux TTC")
ax[2].hist(df_t[df_t.confirmed].Z.clip(upper=100), bins=50, color="gray"); ax[2].set_xlabel("Z des cibles confirmées (m)"); ax[2].set_title("Plage de distances observées")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_coherence.png", bbox_inches="tight"); plt.show()

## 10. Temps réel et faisabilité embarquée (ECU)
Les temps sont mesurés **par étage**. Pour un ECU, ce qui compte est la **latence de bout en bout** et ses **queues de distribution** (P95 / P99), pas la moyenne : une image lente au mauvais moment retarde le freinage.

In [ ]:
#@video
import torch
dev = "GPU : " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Matériel :", dev, "| modèle :", detector.name)
budget = 1000 / meta["fps"]
rows = []
for lab, col in (("Détection + suivi", "t_det_ms"), ("Estimation + décision", "t_est_ms"), ("Latence ECU (det+est)", "t_ecu_ms")):
    s = df[col].iloc[5:]                                     # on écarte l'échauffement CUDA
    rows.append(dict(étage=lab, moyenne_ms=s.mean(), P50=s.quantile(.5), P95=s.quantile(.95), P99=s.quantile(.99), max=s.max(), FPS_moy=1000 / s.mean()))
LAT = pd.DataFrame(rows).round(2)
(display if display else print)(LAT)
print(f"Budget temps réel à {meta['fps']:.0f} fps : {budget:.1f} ms/image ; images hors budget (ECU) : {(df.t_ecu_ms.iloc[5:] > budget).mean()*100:.1f} %")
print(f"Pipeline avec rendu + écriture : {(df.t_ecu_ms + df.t_draw_ms + df.t_write_ms).iloc[5:].mean():.1f} ms/image")
plt.figure(figsize=(8, 3.5)); plt.hist(df.t_ecu_ms.iloc[5:], bins=50, color="steelblue")
plt.axvline(budget, color="r", ls="--", label=f"budget {budget:.0f} ms"); plt.xlabel("Latence ECU par image (ms)"); plt.legend()
plt.savefig(f"{OUT_DIR}/fig_latence.png", bbox_inches="tight"); plt.show()
LAT.to_csv(f"{OUT_DIR}/latences.csv", index=False)

In [ ]:
#@video
# Comparaison des modèles (détection seule, 120 images) : compromis vitesse / nombre de détections
RUN_BENCH = True
if RUN_BENCH:
    cap = cv2.VideoCapture(VIDEO_PATH); frames = []
    while len(frames) < 120:
        ok, f_ = cap.read()
        if not ok: break
        frames.append(f_)
    cap.release(); bench = []
    for w in ("yolov8n.pt", "yolov8s.pt", "yolov8m.pt"):
        d_ = YoloDetector(w, track=False); [d_(frames[0]) for _ in range(3)]     # échauffement
        ts, nd = [], []
        for f_ in frames:
            t0 = time.perf_counter(); r_ = d_(f_); ts.append((time.perf_counter() - t0) * 1000); nd.append(len(r_))
        bench.append(dict(modèle=w, ms_moy=np.mean(ts), P95_ms=np.percentile(ts, 95), FPS=1000 / np.mean(ts), détections_moy=np.mean(nd)))
    BENCH = pd.DataFrame(bench).round(2); (display if display else print)(BENCH); BENCH.to_csv(f"{OUT_DIR}/benchmark_modeles.csv", index=False)

## 11. Synthèse technique
### 11.1 Limites physiques de la vision monoculaire
| Source d'erreur | Mécanisme | Effet mesuré / démontré dans ce notebook |
|---|---|---|
| Hauteur réelle inconnue | $Z=fH/h$ suppose $H$ connue | erreur relative sur Z = erreur relative sur H ; **le TTC reste inchangé** (§4) |
| Quantification / jitter de la boîte | $\sigma_Z/Z=\sigma_h/h$ | l'erreur croît linéairement avec Z (§4), donc avec la distance |
| Boîte tronquée (cible très proche, bord de l'image) | $h_{px}$ sous-estimée → Z surestimée | mesure **écartée**, le filtre prédit seul |
| Route non plane, tangage au freinage | l'angle de visée change | non modélisé ; le TTC par échelle y est moins sensible que Z |
| Dérivée bruitée | $v_{rel}=-\dot Z$ amplifie le bruit | Kalman et régression pondérée réduisent l'erreur de TTC (§4) |
| Intempéries, nuit, éblouissement | contraste ↓, rappel du détecteur ↓ (*domain shift*), réflexions, gouttes | non quantifiable sans données dédiées → fusion nécessaire |
| Latence | détection + filtrage + hydraulique | 100 ms à 50 km/h = 1,4 m parcourus ; **TTC minimal d'évitement** dans §7 |
| Cible hors voie / virage | couloir rectiligne supposé | faux positifs possibles en courbe (voir événements §9.2) |

### 11.2 Apport de la fusion Radar / LiDAR
| Capteur | Force | Faiblesse |
|---|---|---|
| Caméra | classification riche (piéton, vélo, feu) | distance indirecte, sensible météo et lumière |
| Radar 77 GHz | distance **et vitesse (Doppler)** directes, robuste pluie / brouillard / nuit | résolution angulaire faible, classification pauvre |
| LiDAR | nuage 3D précis, forme des objets | coût, dégradé par pluie dense / brouillard |

La fusion (caméra pour *quoi*, radar/LiDAR pour *où* et *à quelle vitesse*) via filtre de Kalman étendu et association de pistes : mesure directe de Z sans hypothèse de hauteur, $v_{rel}$ par Doppler sans dériver un signal bruité, **redondance** (ISO 26262, SOTIF ISO 21448) et moins de freinages fantômes.

### 11.3 Conclusions et perspectives
* Le seuil AEB de 0,8 s **atténue** mais n'évite pas au-delà d'environ 30 km/h (valeur calculée en §7) : un seuil dépendant de la vitesse corrige ce point.
* Les deux estimateurs de TTC, indépendants de H et f, donnent un **contrôle de cohérence** sans capteur supplémentaire (§9.3).
* Perspectives : estimation de la hauteur par classe et dimension de boîte, modèle à accélération constante (TTC de second ordre), calibration réelle de la caméra (distorsion, tangage), évaluation sur dataset annoté en distance (KITTI, nuScenes), détection de voie pour le couloir en courbe.

In [ ]:
#@video
# Export des résultats pour le rapport
df.to_csv(f"{OUT_DIR}/log_images.csv", index=False)
df_t.to_csv(f"{OUT_DIR}/log_cibles.csv", index=False)
MC.groupby(["h_bias", "v_rel"]).size().to_frame("n").to_csv(f"{OUT_DIR}/mc_resume.csv")
T1.to_csv(f"{OUT_DIR}/mc_table_H_exact.csv", index=False); T2.to_csv(f"{OUT_DIR}/mc_table_H_biais.csv", index=False)
json.dump(METRICS, open(f"{OUT_DIR}/metriques.json", "w"), indent=2)
print("Fichiers exportés dans", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

In [ ]:
import os
import shutil

# Dossier de sortie
output_dir = "/kaggle/working/adas_results"

# Créer le dossier
os.makedirs(output_dir, exist_ok=True)

# Afficher les fichiers présents dans le dossier de travail
for root, dirs, files in os.walk("/kaggle/working"):
    if output_dir in root:
        continue

    for file in files:
        if file.endswith((".jpg", ".jpeg", ".png", ".mp4",
                          ".csv", ".json", ".txt", ".pdf")):

            source = os.path.join(root, file)
            destination = os.path.join(output_dir, file)

            # Éviter de copier un fichier sur lui-même
            if os.path.abspath(source) != os.path.abspath(destination):
                shutil.copy2(source, destination)

# Créer l'archive ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/adas_results",
    "zip",
    output_dir
)

print("Archive créée :", zip_path)
print("Taille :", round(os.path.getsize(zip_path) / (1024**2), 2), "Mo")

In [ ]:
# ============================================================
# EXPORT DES RESULTATS DU PROJET AEB/TTC ADAS
# ============================================================

import os
import zipfile
from pathlib import Path
from IPython.display import display, FileLink

# Dossier contenant les résultats Kaggle
output_dir = Path(OUT_DIR)

# Archive finale
zip_path = output_dir / "AEB_TTC_ADAS_Resultats.zip"

# Types de fichiers à récupérer
extensions = {
    ".csv", ".json", ".png", ".jpg", ".jpeg",
    ".mp4", ".pdf", ".txt", ".xlsx"
}

fichiers_exportes = []

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for fichier in sorted(output_dir.iterdir()):

        # Ignorer les dossiers et l'archive elle-même
        if not fichier.is_file() or fichier == zip_path:
            continue

        # Ajouter les résultats pertinents
        if fichier.suffix.lower() in extensions:
            archive.write(fichier, arcname=fichier.name)
            fichiers_exportes.append(fichier.name)

# Afficher le bilan
print("=" * 55)
print("EXPORT DU PROJET ADAS TERMINE")
print("=" * 55)

for nom in fichiers_exportes:
    print("OK :", nom)

print(f"\nNombre de fichiers : {len(fichiers_exportes)}")
print(f"Taille du ZIP : {zip_path.stat().st_size / (1024**2):.2f} Mo")

# Afficher un lien de téléchargement dans le notebook
display(FileLink(str(zip_path)))